# 1. Archiving

**RadDB** turns radar volumes into a compact, queryable Parquet archive. A volume
is an [xarray](https://docs.xarray.dev/) `DataTree` (one group per sweep).

This notebook covers:

1. Raw data and RadDB initialisation
2. Archiving
3. Archived data

---
## How the archive is stored

A radar data is stored as **static data (LUT)** (per-gate geometry, computed once) plus
**dynamic data, one file per volume** (polarimetric variables), linked by an integer `gate_id`.
Following is an example of one archived volume (paths and files):

```
{archive_dir}/{radar}/LUT/{radar}_LUT.parquet          # gate centroids
{archive_dir}/{radar}/LUT/{radar}_h_plane_LUT.parquet  # horizontal gate plane (for PPI)
{archive_dir}/{radar}/LUT/{radar}_v_plane_LUT.parquet  # vertical gate plane  (for RHI)
{archive_dir}/{radar}/LUT/{radar}_corners_LUT.parquet  # 3-D gate corners
{archive_dir}/{radar}/LUT/{radar}_info.yaml            # site, CRS, scan geometry

{archive_dir}/{radar}/{YYYY}/{MM}/{DD}/{radar}_{YYYYMMDD}_{HHMMSS}_POL.parquet    # dynamic data
```

The geometry is stored **once**, not once per volume — which is what keeps the
archive small. Gates with no echo are dropped at archive time (`DBZH > 0` by
default).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import raddb
from raddb.lut import suggest_crs
from pathlib import Path

print("raddb", raddb.__version__)

## 1. Raw data and RadDB initialisation

### Input paths

`MCH_DIR` and `NEXRAD_DIR` hold the DataTree volumes to be archived; `ARCHIVE_DIR`
is where RadDB writes the archive. Edit them to match your own machine.

In [ ]:
# --------------------------------------------------------------------------
# CONFIGURATION — point these at your own data
# --------------------------------------------------------------------------
# Any xarray DataTree with the standard xradar layout works.  
# These tutorials use MeteoSwiss and NEXRAD volumes stored as zarr and nc format.  
# Edit the paths below to point at your own data.

MCH_DIR     = Path("~/Desktop/LTE_project/ltenas8/data/RADAR/MCH_datatree_zarr").expanduser()
NEXRAD_DIR  = Path("~/Desktop/LTE_project/ltenas8/data/RADAR/NEXRAD_datatree_zarr").expanduser()
ARCHIVE_DIR = Path("~/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive").expanduser()

print("MCH DataTrees   :", MCH_DIR)
print("NEXRAD DataTrees:", NEXRAD_DIR)
print("Archive         :", ARCHIVE_DIR)

### Inspecting raw data archive

`inventory(datatree_dir=...)` scans a directory of DataTree files and prints what
it finds: the radar name taken from each filename prefix, the number of files, the
time span they cover and their total size on disk.

In [ ]:
db = raddb.RadDB()
db.inventory(datatree_dir=NEXRAD_DIR)
# db.inventory(datatree_dir=MCH_DIR)

In [ ]:
# `detailed=True` adds a per-day breakdown
db.inventory(datatree_dir=NEXRAD_DIR, detailed=True)
# db.inventory(datatree_dir=MCH_DIR, detailed=True)

### Creating the RadDB object

`RadDB(archive_dir=..., crs=...)` returns an *archive-bound* RadDB: it knows where
the archive lives and which projection to write it in. This is the object used to
archive, open and inspect data.

In [ ]:
db = raddb.RadDB(archive_dir=ARCHIVE_DIR, crs=2056)   # 2056 ==> CH1903+/LV95
db

In [ ]:
db_us = raddb.RadDB(archive_dir=ARCHIVE_DIR)
db_us

## 2. Archiving

`archive()` takes either a directory of DataTree files or an in-memory DataTree.
The LUT is generated automatically from the first volume of each radar.

In [ ]:
result = db.archive(datatree_dir=MCH_DIR, time_period=("2024-06-01", "2024-06-15"))
result

### The CRS constraint

**A projection is mandatory to write an archive, and never needed to read one. Can be given when RadDB is initilaized or at archiving time.**

The LUT stores projected gate coordinates, and every crop and cross-section is
computed in them. A wrong projection is therefore silently wrong: EPSG:2056 (Swiss
LV95) used outside Switzerland mis-measures distance, while the
results could still look perfectly normal. RadDB has no default, the CRS is stated once,
when the object is created, and is checked against the radar's real position before
anything is written.

---

#### Example for US:
A projection is only valid for one large region, so for US radars is chosen per radar . `KTLX` sits in
UTM zone 14N; `KLOT` and `KMLB` are in zones 16N and 17N and would be refused with
that CRS.

In [ ]:
# check what's the suggested crs of the 3 US radars in the NEXRAD dataset
# "KTLX": lat/lon = 35.333/-97.278
# "KMLB": lat/lon = 28.113/-80.654
# "KLOT": lat/lon = 41.604/-88.084
print(f"KTLX ==> {suggest_crs(latitude=35.333, longitude=-97.278)}")
print(f"KMLB ==> {suggest_crs(latitude=28.113, longitude=-80.654)}")  
print(f"KLOT ==> {suggest_crs(latitude=41.604, longitude=-88.084)}")  

In [ ]:
db_us.archive(datatree_dir=NEXRAD_DIR, radar=["KTLX"], crs=32614,  # 32614 ==> UTM zone 14N
              time_period=("2024-01-01", "2024-06-15"))
db_us.archive(datatree_dir=NEXRAD_DIR, radar=["KMLB"], crs=32617,  # 32617 ==> UTM zone 17N
              time_period=("2024-01-01", "2024-06-15"))
db_us.archive(datatree_dir=NEXRAD_DIR, radar=["KLOT"], crs=32616,  # 32616 ==> UTM zone 16N
              time_period=("2024-01-01", "2024-06-15"))

## 3. Archived data

In [ ]:
db = raddb.RadDB(archive_dir=ARCHIVE_DIR)
print("radars in the archive:", db.list_radars())
db.inventory()

In [ ]:
for p in sorted((ARCHIVE_DIR / "L" / "LUT").iterdir()):
    print(f"  {p.name:<28} {p.stat().st_size / 1e6:8.2f} MB")

### `gate_id`: how a volume finds its geometry

One int64 per gate links a row of data (polarimetric variables) to its row of geometry:

```
gate_id = radar_code * 10^12 + sweep * 10^10 + azimuth*10 * 10^6 + range_m
```

In [ ]:
lut = db.get_lut("L")
print("\nLUT:", lut.shape)
print(lut.columns)
print(lut.head(5).select(["gate_id", "sweep", "azimuth", "range", "latitude", "longitude", "altitude"]))
print(lut.head(5).select(["x", "y", "z", "x_2056", "y_2056"]))

### Radar site metadata

`info.yaml` records everything needed to reconstruct the geometry — including the
CRS that was validated at archive time, and the radar's **scan strategy**.

In [ ]:
info = db.get_radar_info("L")
for k in ["radar", "network", "latitude", "longitude", "altitude",
          "crs", "ke", "beamwidth_deg", "n_sweeps", "n_gates"]:
    print(f"  {k:<16} {info[k]}")

---
**Next:** [2 — Opening and filtering](02_opening_and_filtering.ipynb)